In [1]:
!pip install -U "langchain-openrouter"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.2/337.2 kB 7.9 MB/s eta 0:00:00ta 0:00:01


In [2]:
pip install -U "langchain"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
    Uninstalling langgraph-1.1.4:
      Successfully uninstalled langgraph-1.1.4
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.14
    Uninstalling langchain-1.2.14:
      Successfully uninstalled langchain-1.2.14


In [3]:
print("Hello World")

Hello World


In [4]:
import os
import getpass

# Set API key using secure input
api_key = getpass.getpass("Enter your OPENROUTER_API_KEY: ")
os.environ["OPENROUTER_API_KEY"] = api_key
print(f"API key set: {'*' * len(api_key)}")


API key set: *************************************************************************


In [ ]:
from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model="auto")

In [6]:
model.invoke("Hello")

AIMessage(content=" Hello! How can I help you today? If you have any questions or need assistance with something, feel free to ask. I'm here to help.\n\nIf you're just saying hello as a friendly greeting, then hello back to you! Have a great day. 😊", additional_kwargs={}, response_metadata={'model_name': 'mistralai/mistral-7b-instruct-v0.1', 'id': 'gen-1776064999-icgL3IcYjDtDGVhyF1sP', 'created': 1776065000, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019d85b9-2f3e-74d1-98ec-ac54faf0b50b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 63, 'total_tokens': 72, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})

In [ ]:
# initiate open router model

import os
import getpass
from langchain_openrouter import ChatOpenRouter

# Set API key using secure input
api_key = getpass.getpass("Enter your OPENROUTER_API_KEY: ")
os.environ["OPENROUTER_API_KEY"] = api_key
print(f"API key set: {'*' * len(api_key)}")


model = ChatOpenRouter(model="auto")



Good—this is one of the **highest leverage skills in LLM engineering**. If you master few-shot prompting properly, you can often outperform fine-tuning for many real-world tasks.

I’ll take you from **theory → mechanisms → design patterns → failure modes → experiments → pro-level tasks**.

---

# 🧠 1. What is Few-Shot Prompting (Deep View)

Few-shot prompting = **teaching the model a task using examples inside the prompt itself**.

Instead of:

```
Translate to French: Hello
```

You do:

```
English: Hello → French: Bonjour
English: Thank you → French: Merci
English: Good night → French:
```

👉 The model **infers the pattern** (this is called *in-context learning*).

---

## ⚠️ Key Insight (IMPORTANT)

The model is NOT “learning” weights.

It is:

* Pattern matching
* Distribution alignment
* Conditional generation based on examples

Think of it as:

> “simulate the function demonstrated in examples”

---

# ⚙️ 2. Types of Few-Shot

## 🔹 1-shot

One example

* Fast, cheap
* Weak generalization

## 🔹 2-shot

Two examples

* Better pattern clarity
* Still fragile

## 🔹 N-shot (3–10 typical)

* Stronger consistency
* Higher token cost
* Risk of overfitting to examples

---

## 🔬 Rule of Thumb

| Task Type                     | Shots Needed |
| ----------------------------- | ------------ |
| Simple formatting             | 1–2          |
| Classification                | 3–5          |
| Complex reasoning             | 5–10         |
| Structured output (JSON/code) | 3–8          |

---

# 🧪 3. Why Few-Shot Works (Mechanism)

Internally, the model:

1. Reads examples
2. Detects:

   * Input pattern
   * Output format
   * Transformation rule
3. Applies same transformation

👉 This is similar to **meta-learning behavior**

---

# 🧱 4. Anatomy of a High-Quality Few-Shot Prompt

### Structure

```
[Instruction]
[Examples]
[Query]
```

---

## ✅ Example (Professional Level)

```
Classify sentiment as Positive, Negative, or Neutral.

Review: "I love this product!" → Sentiment: Positive
Review: "Worst experience ever." → Sentiment: Negative
Review: "It's okay, not great." → Sentiment: Neutral

Review: "The design is nice but performance is slow." → Sentiment:
```

---

## 🔥 Critical Design Rules

### 1. Consistent formatting (NON-NEGOTIABLE)

Bad:

```
Text: Hello → French: Bonjour
Hello = Bonjour
```

Good:

```
English: X → French: Y
```

---

### 2. Use delimiters

```
### Example 1
Input: ...
Output: ...
```

👉 Helps model parse structure cleanly

---

### 3. Match real distribution

Examples should reflect:

* Real-world inputs
* Edge cases
* Ambiguity

---

# 🧠 5. Example Selection Strategy (ADVANCED)

This is where most people fail.

---

## 🔹 Strategy 1: Diversity Sampling

Include:

* Easy case
* Hard case
* Edge case

Example:

```
"I love it" → Positive
"I hate it" → Negative
"It's fine" → Neutral
```

---

## 🔹 Strategy 2: Boundary Cases

Very important for classification:

```
"A bit slow but usable" → Neutral
"Absolutely terrible" → Negative
```

---

## 🔹 Strategy 3: Similarity-Based (PRO LEVEL)

Pick examples **closest to user query**

👉 Used in RAG + dynamic prompting

---

## 🔹 Strategy 4: Failure-Based Selection

Include examples where model usually fails

---

# 🔁 6. Order Effects (VERY IMPORTANT)

Changing order of examples can change output.

---

## 🔬 Example

```
Positive → Negative → Neutral
```

vs

```
Neutral → Negative → Positive
```

👉 Model bias shifts

---

## 🧠 Why?

* Recency bias (later examples influence more)
* Pattern anchoring

---

## ✅ Best Practice

* Put **most important example last**
* Keep logical progression

---

# 🧪 7. Formatting Patterns (Production Level)

---

## 🔹 Pattern 1: Arrow Format

```
Input → Output
```

✔ Fast
❌ Weak for complex tasks

---

## 🔹 Pattern 2: Key-Value Format

```
Input: ...
Output: ...
```

✔ Best general-purpose

---

## 🔹 Pattern 3: JSON Format (PRO)

```
{
  "input": "...",
  "output": "..."
}
```

✔ Best for APIs / structured output

---

## 🔹 Pattern 4: Chat Format

```
User: ...
Assistant: ...
```

✔ Best for conversational systems

---

# 🚨 8. Failure Modes

---

## ❌ 1. Overfitting to examples

Model copies patterns too literally

---

## ❌ 2. Format drift

Output format breaks

---

## ❌ 3. Example leakage

Model reuses example text

---

## ❌ 4. Too many examples

* Context overload
* Performance drops

---

# 🧪 9. Zero-shot vs Few-shot

---

## Zero-shot

```
Classify sentiment: "I love this"
```

✔ Cheap
❌ Inconsistent

---

## Few-shot

✔ More consistent
✔ Better formatting
❌ More tokens

---

## 🔬 Real Insight

Few-shot ≈ **soft fine-tuning inside context**

---

# 🧪 10. Pro-Level Experiment Framework

You MUST test, not guess.

---

## Experiment Design

### Step 1: Create dataset

10–20 inputs

---

### Step 2: Test:

* Zero-shot
* 1-shot
* 3-shot
* 5-shot

---

### Step 3: Measure:

| Metric           | Meaning              |
| ---------------- | -------------------- |
| Accuracy         | Correctness          |
| Consistency      | Same output repeated |
| Format adherence | Valid JSON etc       |

---

# 💻 11. Colab Experiment Code (Core)

```python
from openai import OpenAI
client = OpenAI()

def run_prompt(prompt):
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content


zero_shot = "Classify sentiment: I love this product"
few_shot = """
Classify sentiment as Positive, Negative, Neutral.

Review: I love this → Positive
Review: I hate this → Negative
Review: It's okay → Neutral

Review: I really enjoyed this →
"""

print("Zero-shot:", run_prompt(zero_shot))
print("Few-shot:", run_prompt(few_shot))
```

---

# 🔥 12. PRO MAX TASKS (DO THESE SERIOUSLY)

---

## 🧩 Task 1: Build 5 Few-Shot Prompts

Create prompts for:

1. Sentiment classification
2. Email tone detection
3. Code bug detection
4. Bengali → English translation
5. JSON extraction

👉 Each must have:

* 3 examples
* Consistent format

---

## 🧪 Task 2: Compare with Zero-Shot

For each task:

* Run without examples
* Run with few-shot

👉 Measure:

* Accuracy
* Output format quality

---

## 🔁 Task 3: Order Effect Experiment

Take same examples:

* Shuffle order
* Observe output differences

👉 Write conclusion:

* What changed?
* Why?

---

## 🧠 Task 4: Example Optimization (ADVANCED)

Take a bad prompt and improve:

* Add edge cases
* Improve formatting
* Reorder examples

---

## 🧪 Task 5: Token Efficiency Challenge

Goal:

* Reduce prompt size by 30%
* Keep same performance

---

## 💣 Task 6 (PRO MAX)

Build:

### “Dynamic Few-Shot System”

Steps:

1. Store 50 examples
2. Use similarity (cosine / embeddings)
3. Select top 3 relevant examples
4. Inject into prompt

👉 This is how real AI systems work (RAG + few-shot)

---

# 🚀 What You’ll Gain

If you complete all tasks:

* You’ll outperform 90% prompt engineers
* You’ll understand LLM behavior deeply
* You’ll be ready for:

  * RAG systems
  * AI SaaS products
  * Production LLM pipelines

---

If you want next level, I can teach you:

👉 **Chain-of-Thought vs Few-Shot vs Tool Use (when to use which)**
👉 **Automatic prompt optimization (AutoPrompt / DSPy style)**
👉 **Few-shot + RAG hybrid architecture**

Just tell me.


In [ ]:


def run_prompt(prompt):
    response = model.invoke(prompt)
    return response.content


zero_shot = "Classify sentiment: I love this product"
few_shot = """
Classify sentiment as Positive, Negative, Neutral.

Review: I love this → Positive
Review: I hate this → Negative
Review: It's okay → Neutral

Review: I really enjoyed this →
"""

print("Zero-shot:", run_prompt(zero_shot))
print("Few-shot:", run_prompt(few_shot))